# TradeSense · Component 03 · Stage D · Notebook D2
### Train the TGNN (M1: with order-book graph edges) and its ablation (M0: no neighbour edges), plus a matched MLP (B3a). Validation month only.

**What this notebook does.** It trains, on the training months (May, June 2024), three learned models that all use the same input window, the same output rule (*value at t plus a learned change*, starting exactly at persistence), the same loss (L1 on the next-interval values + 0.2 × binary cross-entropy for "will this node change?") and the same training budget:

| model | what it is | graph edges |
|---|---|---|
| **M1** | the TGNN: 2 message-passing layers over 51 nodes (25 bid levels, 25 ask levels, 1 global node) + a GRU over the 50 past intervals | rank chain (bid 1-2-…-25, ask 1-2-…-25), best-bid ↔ best-ask, every node ↔ global |
| **M0** | the ablation: the same network but **without** the neighbour edges (a node sees only itself and the global node) | global only |
| **B3a** | the D1 MLP (same size as the best D1 configuration) with the same extra output and loss | none (flattened window) |

M1 and M0 are tuned on the same 2 configurations (hidden 64 × learning rate 1e-3 or 3e-4, one seed; the budget was reduced from 4 configurations and 20 epochs to 2 configurations and 12 epochs after the first runs measured about 6 minutes per epoch; this was decided before any test day was opened and is recorded in the protocol), then trained with 3 seeds each; B3a uses the best D1 configuration with 3 seeds. Early stopping and configuration choice use a **fixed 25% random subset of the validation windows** (the same windows for every model); the final scoring uses **all** validation windows. The scores are compared with B1 (persistence) and B3 from D1 — *descriptively*. The pass/fail tests of H1/H2 happen once, on the test days, in D3.

**What it does NOT do.** It opens no August, September or October day. It changes no frozen rule. It writes into the new folder `TGNN_StageD_v1/d2` (nothing existing is overwritten).

**Honest expectation (recorded in the protocol before this run).** D1b found small headroom (the best simple predictor beat persistence by about 1-2% under the all-target MAE). The 5% rule of H1/H2 is therefore likely to fail or be marginal. If so it is reported as failed. The TGNN is not required to outperform the three completed models; its value is the relational order-book view and its irregularity score.

**Use a T4 GPU runtime.** Total training is long (roughly 10-12 GPU hours in all). The notebook stops itself after `MAX_RUN_MINUTES`, keeps every finished epoch, and **continues where it stopped when you press *Run all* again** (it can also be continued on another Google account if you copy the `TGNN_StageD_v1` folder there). The last cell prints the SHARE BLOCK only when everything is finished.

## 1 · Settings (nothing to edit; `MAX_RUN_MINUTES` can be shortened or lengthened)

In [2]:
import os, gc, sys, json, time, shutil, platform, warnings
from pathlib import Path
import numpy as np, pandas as pd
warnings.filterwarnings('ignore')
from IPython.display import display
T_START = time.time()
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Not in Colab -> using TRADESENSE_BASE.')
BASE = Path(os.environ.get('TRADESENSE_BASE', '/content/drive/MyDrive/research datasets')); assert BASE.is_dir(), f'Drive folder not found: {BASE}'
STAGEC_NAME = 'TGNN_StageC_GraphFeatures_v2'; D1_NAME = 'TGNN_StageD_v1'
MAX_RUN_MINUTES = 170        # the notebook stops cleanly after this many minutes of training (counted from the start of the training cell) (an epoch in progress is finished first); press Run all again to continue
# ---- budget and rules, fixed BEFORE the run (Protocol v2 sections 14.3 and 14.9) ----
FRAC = 0.10; MAX_EPOCHS = 12; PATIENCE = 3; BATCH = 128; AUX = 0.2; VAL_FRAC = 0.25; AMP = True
SEED_TUNE = 200; SEEDS = [201, 202, 203]
GRID = [{'hidden': 64, 'lr': lr} for lr in (1e-3, 3e-4)]      # reduced from 4 to 2 configurations after the first tuning runs showed ~6 min per epoch (decided before any test day was opened)
AUC_WINDOWS = 60_000         # validation windows used for the change-occurrence AUC (same windows for every model)
EXPECTED = {'train_windows': 3453034, 'val_windows': 1725989}
print('budget:', dict(frac=FRAC, max_epochs=MAX_EPOCHS, patience=PATIENCE, batch=BATCH, aux=AUX, val_frac=VAL_FRAC, amp=AMP, seeds=SEEDS), '| max run minutes', MAX_RUN_MINUTES)

Mounted at /content/drive
budget: {'frac': 0.1, 'max_epochs': 12, 'patience': 3, 'batch': 128, 'aux': 0.2, 'val_frac': 0.25, 'amp': True, 'seeds': [201, 202, 203]} | max run minutes 170


## 2 · Libraries (written to the Colab working folder, not to Drive)

`staged_lib.py` is the exact D1 library (its hash is checked against D1 in cell 3). `staged_diag_lib.py` is the D1b helper (only its AUC and test-data generator are used). `staged_d2_lib.py` holds the TGNN, the ablation, the matched MLP and the resumable trainer.

In [3]:
%%writefile staged_lib.py
"""TradeSense C03 - Stage D library (D1: loader, leak guards, baselines B1-B4, B3 MLP, resumable trainer).
Reads ONLY the frozen Stage C tensors. Never writes to Stage C. Test days are refused until a freeze file exists."""
import os, json, time, math, hashlib, shutil
from pathlib import Path
import numpy as np
import torch, torch.nn as nn

SEQ = 50; NN = 50; NCH = 6; GCH = 3; NT = 51; LOGQ = 2; GLS = 0; GIMB = 1
SYMS = ['BTCUSDT', 'ETHUSDT']
SPLITS = {'train': ['2024-05-01', '2024-06-01'], 'val': ['2024-07-01'], 'test': ['2024-08-01', '2024-09-01', '2024-10-01']}
def keys_of(split): return [f'{s}_{d}' for d in SPLITS[split] for s in SYMS]

# ---------------------------------------------------------------- small helpers
def sha256_file(p, chunk=16 << 20):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        while True:
            b = f.read(chunk)
            if not b: break
            h.update(b)
    return h.hexdigest()
def _j(o):
    if isinstance(o, (np.integer,)): return int(o)
    if isinstance(o, (np.floating,)): return float(o)
    if isinstance(o, np.ndarray): return o.tolist()
    if isinstance(o, Path): return str(o)
    raise TypeError(type(o))
def canonical_sha(obj): return hashlib.sha256(json.dumps(obj, sort_keys=True, default=_j).encode()).hexdigest()
def dump_json(obj, path):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True); tmp = path.with_name(path.name + '.tmp')
    tmp.write_text(json.dumps(obj, indent=1, default=_j)); os.replace(tmp, path)
def put_json(obj, path):
    """write once; if a different file already exists it is NEVER replaced: the new content goes to <name>.rN.json"""
    path = Path(path)
    if path.exists():
        if json.loads(path.read_text()) == json.loads(json.dumps(obj, default=_j)): return path
        n = 1
        while path.with_name(f'{path.stem}.r{n}{path.suffix}').exists(): n += 1
        path = path.with_name(f'{path.stem}.r{n}{path.suffix}')
    dump_json(obj, path); return path
def check_frozen(path, cfg):
    """first call writes the frozen config; later calls must match exactly, otherwise resuming is refused"""
    path = Path(path)
    if not path.exists(): dump_json({'config': cfg, 'sha256': canonical_sha(cfg)}, path); return 'written'
    old = json.loads(path.read_text())
    if old['sha256'] != canonical_sha(cfg): raise RuntimeError(f'frozen config differs from the saved one ({path}); do not change settings or code while resuming a run')
    return 'matches'

# ---------------------------------------------------------------- the loader (all tensors in one block, optionally on the GPU)
class Bank:
    def __init__(self, stagec_dir, keys, device='cpu', freeze_file=None, chunk=100_000):
        self.dir = Path(stagec_dir) / 'sets'; self.keys = list(keys); self.device = device; self.summ = {}; total = 0
        for k in self.keys:
            s = json.loads((self.dir / k / 'set_summary.json').read_text()); self.summ[k] = s
            if s['split'] == 'test' and not (freeze_file and Path(freeze_file).exists()):
                raise PermissionError(f'{k} is a TEST day: refused because the freeze file does not exist yet')
            total += int(s['n_intervals'])
        self.nodes = torch.empty((total, NN, NCH), dtype=torch.float16, device=device); self.glob = torch.empty((total, GCH), dtype=torch.float16, device=device)
        self.off = {}; wins = []; sid = []; o = 0
        for i, k in enumerate(self.keys):
            arr = np.load(self.dir / k / 'nodes.npy', mmap_mode='r'); g = np.load(self.dir / k / 'glob.npy', mmap_mode='r'); n = arr.shape[0]
            assert arr.shape[1:] == (NN, NCH) and g.shape == (n, GCH), f'{k}: unexpected tensor shape {arr.shape} {g.shape}'
            for a in range(0, n, chunk):
                b = min(a + chunk, n); self.nodes[o + a:o + b] = torch.from_numpy(np.array(arr[a:b])).to(device)
            self.glob[o:o + n] = torch.from_numpy(np.array(g)).to(device)
            w = np.load(self.dir / k / 'win_end.npy'); wins.append(w.astype(np.int64) + o); sid.append(np.full(len(w), i, np.int16)); self.off[k] = (o, n); o += n; del arr, g
        self.win = np.concatenate(wins); self.win_set = np.concatenate(sid); self.ar = torch.arange(-SEQ + 1, 1, device=device)
    def gb(self): return (self.nodes.numel() * 2 + self.glob.numel() * 2) / 1e9
    def batch(self, g):
        """g: LongTensor of GLOBAL end indices t. Input = intervals t-49..t (inclusive). Target = interval t+1 only."""
        idx = g[:, None] + self.ar
        xn = self.nodes[idx].float(); xg = self.glob[idx].float()
        yn = self.nodes[g + 1][:, :, LOGQ].float(); yg = self.glob[g + 1][:, GLS].float()
        y = torch.cat([yn, yg[:, None]], 1); last = torch.cat([xn[:, -1, :, LOGQ], xg[:, -1, GLS:GLS + 1]], 1)
        return xn, xg, y, last

# ---------------------------------------------------------------- evaluation harness (per-window losses are kept for the later bootstrap)
@torch.no_grad()
def evaluate(bank, predict, bs=4096, extra=None):
    n = len(bank.win); loss = np.zeros(n, np.float32); chg = np.zeros(n, np.float32); nch = np.zeros(n, np.int16); spr = np.zeros(n, np.float32); ex = []
    for a in range(0, n, bs):
        b = min(a + bs, n); g = torch.from_numpy(bank.win[a:b]).to(bank.device); xn, xg, y, last = bank.batch(g)
        err = (predict(xn, xg, last) - y).abs(); m = (y != last)
        loss[a:b] = err.mean(1).cpu().numpy(); chg[a:b] = (err * m).sum(1).cpu().numpy(); nch[a:b] = m.sum(1).cpu().numpy(); spr[a:b] = err[:, NN].cpu().numpy()
        if extra is not None: ex.append(extra(xn, xg, last).cpu().numpy())
    out = {'loss': loss, 'chg_sum': chg, 'n_chg': nch, 'spr': spr}
    if extra is not None: out['extra'] = np.concatenate(ex)
    return out
def summarize(bank, res):
    rows = {}
    for i, k in enumerate(bank.keys):
        m = bank.win_set == i
        rows[k] = {'n_windows': int(m.sum()), 'mae_all': float(res['loss'][m].mean()), 'mae_changed': float(res['chg_sum'][m].sum() / max(int(res['n_chg'][m].sum()), 1)),
                   'share_changed': float(res['n_chg'][m].sum() / (NT * max(int(m.sum()), 1))), 'mae_spread': float(res['spr'][m].mean())}
    return rows
def average_results(rs):
    """seeds are averaged FIRST (mean of the per-seed losses of each window), exactly as in the frozen rule"""
    return {'loss': np.mean([r['loss'] for r in rs], 0).astype(np.float32), 'chg_sum': np.mean([r['chg_sum'] for r in rs], 0).astype(np.float32),
            'n_chg': rs[0]['n_chg'], 'spr': np.mean([r['spr'] for r in rs], 0).astype(np.float32)}
def save_result(res, bank, path):
    path = Path(path)
    if path.exists(): raise FileExistsError(path)
    tmp = path.with_name(path.stem + '.tmp.npz'); np.savez_compressed(tmp, win=bank.win, win_set=bank.win_set, **{k: v for k, v in res.items() if k != 'extra'}); os.replace(tmp, path)

# ---------------------------------------------------------------- baselines
def pred_b1(xn, xg, last): return last
def ema_weights(span, device='cpu'):
    a = 2.0 / (span + 1.0); j = torch.arange(SEQ - 1, -1, -1, dtype=torch.float32, device=device)       # time index 0 = oldest, 49 = latest
    w = (1 - a) ** j; return w / w.sum()
SPANS = [1, 2, 3, 5, 8, 13, 21, 34, 50]
def make_b2(span_node, span_spread):
    def f(xn, xg, last):
        wn = ema_weights(span_node, xn.device); ws = ema_weights(span_spread, xn.device)
        pn = torch.einsum('t,btn->bn', wn, xn[:, :, :, LOGQ]); ps = torch.einsum('t,bt->b', ws, xg[:, :, GLS])
        return torch.cat([pn, ps[:, None]], 1)
    return f
@torch.no_grad()
def tune_b2(bank, bs=4096):
    """one span for node targets, one span for the spread target, chosen on validation only"""
    sn = np.zeros(len(SPANS)); ss = np.zeros(len(SPANS)); n = len(bank.win)
    for a in range(0, n, bs):
        b = min(a + bs, n); g = torch.from_numpy(bank.win[a:b]).to(bank.device); xn, xg, y, last = bank.batch(g)
        for i, sp in enumerate(SPANS):
            p = make_b2(sp, sp)(xn, xg, last); e = (p - y).abs(); sn[i] += float(e[:, :NN].sum()); ss[i] += float(e[:, NN].sum())
    sn /= n * NN; ss /= n; return {'span_node': SPANS[int(sn.argmin())], 'span_spread': SPANS[int(ss.argmin())], 'val_mae_node_by_span': dict(zip(SPANS, sn.tolist())), 'val_mae_spread_by_span': dict(zip(SPANS, ss.tolist()))}
def pred_b4(xn, xg, last):
    """B4: the z-score centre as a naive predictor. Spread target -> window mean (z = 0). Node targets are not part of B4 (persistence is used only so that a tensor exists; B4 is judged on the spread column only)."""
    return torch.cat([last[:, :NN], xg[:, :, GLS].mean(1, keepdim=True)], 1)
def b4_score(xn, xg, last):
    """rolling z-score anomaly score of spread and top-5 imbalance (max of the two absolute z), saved for later fusion/anomaly use"""
    def z(v): return ((v[:, -1] - v.mean(1)) / (v.std(1) + 1e-3)).abs()
    return torch.maximum(z(xg[:, :, GLS]), z(xg[:, :, GIMB]))

# ---------------------------------------------------------------- B3 MLP (the same output rule is used for every learned model: prediction = value at t + learned change)
class MLP(nn.Module):
    def __init__(s, h):
        super().__init__(); s.net = nn.Sequential(nn.Linear(SEQ * NN * NCH + SEQ * GCH, h), nn.ReLU(), nn.Linear(h, h), nn.ReLU(), nn.Linear(h, NT))
        nn.init.zeros_(s.net[-1].weight); nn.init.zeros_(s.net[-1].bias)           # starts exactly at persistence
    def forward(s, xn, xg, last): return last + s.net(torch.cat([xn.flatten(1), xg.flatten(1)], 1))
def make_mlp(cfg): return MLP(cfg['hidden'])
GRID = [{'hidden': h, 'lr': lr} for h in (64, 128) for lr in (1e-3, 3e-4)]      # the 4 pre-listed configurations

# ---------------------------------------------------------------- resumable trainer (also used by D2 for the TGNN)
def _load_latest(out, model, opt):
    for p in sorted(out.glob('resume_e*.pt'), reverse=True):
        try: ck = torch.load(p, map_location='cpu', weights_only=False)
        except Exception: continue
        model.load_state_dict(ck['model']); opt.load_state_dict(ck['opt']); torch.set_rng_state(ck['rng_cpu'])
        if ck.get('rng_cuda') is not None and torch.cuda.is_available(): torch.cuda.set_rng_state_all(ck['rng_cuda'])
        return ck['state'], p.name
    return None, None
def fit(make_model, cfg, bank_tr, bank_va, out_dir, seed, frac=0.10, max_epochs=20, patience=3, bs=256, device='cpu', log=print, _halt_after=None):
    out = Path(out_dir); out.mkdir(parents=True, exist_ok=True)
    if (out / 'done.json').exists(): return json.loads((out / 'done.json').read_text())
    assert all(bank_tr.summ[k]['split'] == 'train' for k in bank_tr.keys), 'training bank contains a non-training set'
    torch.manual_seed(seed); model = make_model(cfg).to(device); opt = torch.optim.Adam(model.parameters(), lr=cfg['lr'])
    state = {'epoch': 0, 'best': float('inf'), 'best_epoch': 0, 'bad': 0, 'hist': []}
    st, name = _load_latest(out, model, opt)
    if st is not None: state = st; log(f'  resumed from {name} (epoch {state["epoch"]})')
    n = len(bank_tr.win)
    while state['epoch'] < max_epochs and state['bad'] < patience:
        e = state['epoch'] + 1; t0 = time.time(); rng = np.random.default_rng([seed, e]); pick = rng.permutation(n)[:int(n * frac)]
        model.train(); tot = torch.zeros((), device=device); cnt = 0
        for a in range(0, len(pick), bs):
            g = torch.from_numpy(bank_tr.win[pick[a:a + bs]]).to(device); xn, xg, y, last = bank_tr.batch(g)
            loss = (model(xn, xg, last) - y).abs().mean(); opt.zero_grad(set_to_none=True); loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
            tot += loss.detach() * len(g); cnt += len(g)
        t_train = time.time() - t0; model.eval(); res = evaluate(bank_va, lambda xn, xg, last: model(xn, xg, last)); val = float(res['loss'].mean())
        torch.save(model.state_dict(), out / f'weights_e{e:03d}.pt')
        state['epoch'] = e; state['hist'].append({'epoch': e, 'train_mae': float(tot / max(cnt, 1)), 'val_mae': val, 'train_seconds': round(t_train, 1), 'epoch_seconds': round(time.time() - t0, 1)})
        if val < state['best'] - 1e-9: state['best'] = val; state['best_epoch'] = e; state['bad'] = 0
        else: state['bad'] += 1
        tmp = out / f'resume_e{e:03d}.pt.tmp'
        torch.save({'model': model.state_dict(), 'opt': opt.state_dict(), 'rng_cpu': torch.get_rng_state(), 'rng_cuda': torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None, 'state': state}, tmp); os.replace(tmp, out / f'resume_e{e:03d}.pt')
        for p in out.glob('resume_e*.pt'):                                  # keep the two newest optimiser states; weights of every epoch stay
            if int(p.stem[8:]) < e - 1: p.unlink()
        with open(out / 'log.jsonl', 'a') as f: f.write(json.dumps(state['hist'][-1]) + '\n')
        log(f'  epoch {e}: train {state["hist"][-1]["train_mae"]:.5f} | val {val:.5f} | best {state["best"]:.5f} (e{state["best_epoch"]}) | {state["hist"][-1]["epoch_seconds"]} s')
        if _halt_after is not None and e >= _halt_after: return None
    done = {'cfg': cfg, 'seed': seed, 'best_val': state['best'], 'best_epoch': state['best_epoch'], 'epochs_run': state['epoch'], 'history': state['hist'], 'n_params': int(sum(p.numel() for p in model.parameters())),
            'mean_epoch_seconds': float(np.mean([h['epoch_seconds'] for h in state['hist']])), 'frac': frac, 'max_epochs': max_epochs, 'patience': patience, 'batch_size': bs}
    dump_json(done, out / 'done.json'); return done
def load_best(make_model, cfg, out_dir, device='cpu'):
    d = json.loads((Path(out_dir) / 'done.json').read_text()); m = make_model(cfg).to(device); m.load_state_dict(torch.load(Path(out_dir) / f'weights_e{d["best_epoch"]:03d}.pt', map_location=device)); m.eval(); return m, d

# ---------------------------------------------------------------- synthetic Stage C folder (for the code tests only)
def make_mock_stagec(root, n_int=2600, seed=0, bad_gap=(1200, 1290)):
    root = Path(root); sets = root / 'sets'; (root / 'config').mkdir(parents=True, exist_ok=True); rng = np.random.default_rng(seed)
    dump_json({'scaler': {'mock': True}, 'sha256': canonical_sha({'mock': True})}, root / 'config' / 'scaler_train_only.json')
    for split in ('train', 'val', 'test'):
        for day in SPLITS[split]:
            for sym in SYMS:
                k = f'{sym}_{day}'; d = sets / k; d.mkdir(parents=True, exist_ok=True)
                lq = np.zeros((n_int, NN), np.float32); cur = rng.normal(size=NN)
                for i in range(n_int):
                    ch = rng.random(NN) < 0.25; cur = np.where(ch, 0.9 * cur + rng.normal(scale=0.5, size=NN), cur); lq[i] = cur
                nodes = rng.normal(size=(n_int, NN, NCH)).astype(np.float32); nodes[:, :, 0] = np.where(np.arange(NN) < 25, -1, 1); nodes[:, :, LOGQ] = lq
                gl = rng.normal(size=(n_int, GCH)).astype(np.float32); gl[1:, GLS] = np.where(rng.random(n_int - 1) < 0.2, rng.normal(size=n_int - 1), 0); gl[:, GLS] = np.maximum.accumulate(np.zeros(n_int)) + np.cumsum(np.where(rng.random(n_int) < .1, rng.normal(scale=.3, size=n_int), 0))
                flags = np.zeros(n_int, np.uint8); flags[:SEQ + 20] = 16; flags[bad_gap[0]:bad_gap[1]] = 8
                c = np.concatenate([[0], np.cumsum((flags != 0).astype(np.int64))]); t = np.arange(SEQ, n_int - 1); win = t[(c[t + 2] - c[t - SEQ]) == 0].astype(np.int32)
                np.save(d / 'nodes.npy', nodes.astype(np.float16)); np.save(d / 'glob.npy', gl.astype(np.float16)); np.save(d / 'flags.npy', flags); np.save(d / 'win_end.npy', win)
                dump_json({'sym': sym, 'day': day, 'split': split, 'n_intervals': n_int, 'n_windows': int(len(win)), 'scaler_sha256': canonical_sha({'mock': True}), 'hashes': {f: sha256_file(d / f) for f in ('nodes.npy', 'glob.npy', 'flags.npy', 'win_end.npy')}}, d / 'set_summary.json')
    return root

# ---------------------------------------------------------------- E-tests (code on synthetic data)
def unit_tests(tmp):
    tmp = Path(tmp); out = []
    def add(i, nm, ok, det=''): out.append((i, nm, bool(ok), str(det)))
    root = make_mock_stagec(tmp / 'sc', n_int=2600); fz = tmp / 'freeze.json'
    # E01 test days are refused before the freeze file exists
    try: Bank(root, keys_of('test')[:1], freeze_file=fz); refused = False
    except PermissionError: refused = True
    dump_json({'x': 1}, fz); ok_after = True
    try: Bank(root, keys_of('test')[:1], freeze_file=fz)
    except Exception: ok_after = False
    fz.unlink(); add('E01', 'test days refused until the freeze file exists', refused and ok_after, f'refused={refused} allowed_after={ok_after}')
    btr = Bank(root, keys_of('train'), device='cpu'); bva = Bank(root, keys_of('val'), device='cpu')
    # E02 exact window indexing against direct array access
    rng = np.random.default_rng(1); worst = 0.0; okidx = True
    for i in rng.choice(len(btr.win), 40, replace=False):
        gt = int(btr.win[i]); k = btr.keys[int(btr.win_set[i])]; o, n = btr.off[k]; t = gt - o
        raw = np.load(btr.dir / k / 'nodes.npy'); rg = np.load(btr.dir / k / 'glob.npy')
        xn, xg, y, last = btr.batch(torch.tensor([gt]))
        okidx &= np.array_equal(xn[0].numpy(), raw[t - 49:t + 1].astype(np.float32)) and np.array_equal(xg[0].numpy(), rg[t - 49:t + 1].astype(np.float32))
        okidx &= np.array_equal(y[0, :NN].numpy(), raw[t + 1, :, LOGQ].astype(np.float32)) and float(y[0, NN]) == float(rg[t + 1, GLS]) and np.array_equal(last[0, :NN].numpy(), raw[t, :, LOGQ].astype(np.float32))
    add('E02', 'input = intervals t-49..t, target = interval t+1 only (40 random windows, exact)', okidx)
    # E03 the future cannot leak: overwrite interval t+1 and t+2 and check the input does not change
    gt = int(btr.win[100]); a = btr.batch(torch.tensor([gt]))[0].clone(); y0 = btr.batch(torch.tensor([gt]))[2].clone()
    sv = btr.nodes[gt + 1:gt + 3].clone(); btr.nodes[gt + 1:gt + 3] = 77.0; b = btr.batch(torch.tensor([gt]))
    same_x = torch.equal(a, b[0]); y_changed = not torch.equal(y0, b[2]); btr.nodes[gt + 1:gt + 3] = sv; add('E03', 'changing t+1 and t+2 does not change the input, only the target', same_x and y_changed)
    # E04 no window touches a flagged interval, and no window crosses a day boundary
    okf = True; okb = True
    for i, k in enumerate(btr.keys):
        fl = np.load(btr.dir / k / 'flags.npy'); w = np.load(btr.dir / k / 'win_end.npy'); okf &= all((fl[t - SEQ:t + 2] == 0).all() for t in w[::7]); okb &= int(w.min()) >= SEQ and int(w.max()) + 1 < btr.off[k][1]
    add('E04', 'windows contain no flagged interval and stay inside their day', okf and okb)
    # E05 baselines
    bs_ = 512; res1 = evaluate(bva, pred_b1, bs=bs_); man = []
    for i in rng.choice(len(bva.win), 30, replace=False):
        xn, xg, y, last = bva.batch(torch.tensor([int(bva.win[i])])); man.append((i, float((last - y).abs().mean())))
    okb1 = all(abs(res1['loss'][i] - v) < 1e-6 for i, v in man)
    res2 = evaluate(bva, make_b2(1, 1)); w = ema_weights(8); okb2 = np.allclose(res2['loss'], res1['loss'], atol=1e-6) and abs(float(w.sum()) - 1) < 1e-6 and bool((w >= 0).all()) and bool(w[-1] == w.max())
    add('E05', 'B1 equals a hand computation; B2 with span 1 equals B1; EMA weights are valid and causal', okb1 and okb2)
    # E06 changed-node mask by hand
    xn = torch.zeros(1, SEQ, NN, NCH); xg = torch.zeros(1, SEQ, GCH); last = torch.zeros(1, NT); y = torch.zeros(1, NT); y[0, 3] = 1.0; y[0, 50] = 2.0; p = torch.zeros(1, NT); err = (p - y).abs(); m = (y != last)
    add('E06', 'changed-target mask and per-window sums', int(m.sum()) == 2 and float((err * m).sum()) == 3.0)
    # E07 MLP starts exactly at persistence and can learn
    torch.manual_seed(0); mlp = MLP(16); xn, xg, y, last = btr.batch(torch.tensor(btr.win[:8])); p0 = mlp(xn, xg, last); okz = torch.equal(p0, last)
    opt = torch.optim.Adam(mlp.parameters(), lr=1e-3); l0 = float((mlp(xn, xg, last) - y).abs().mean().detach())
    for _ in range(60): l = (mlp(xn, xg, last) - y).abs().mean(); opt.zero_grad(); l.backward(); opt.step()
    add('E07', 'B3 starts at persistence and its loss falls on a small fit', okz and float(l.detach()) < l0 * 0.9, f'{l0:.4f}->{float(l.detach()):.4f}')
    # E08 resume gives exactly the same result as an uninterrupted run
    cfg = {'hidden': 8, 'lr': 1e-3}; quiet = lambda *a, **k: None
    d1 = fit(make_mlp, cfg, btr, bva, tmp / 'r_full', seed=5, frac=0.2, max_epochs=3, patience=9, bs=64, log=quiet)
    fit(make_mlp, cfg, btr, bva, tmp / 'r_part', seed=5, frac=0.2, max_epochs=3, patience=9, bs=64, log=quiet, _halt_after=1)
    d2 = fit(make_mlp, cfg, btr, bva, tmp / 'r_part', seed=5, frac=0.2, max_epochs=3, patience=9, bs=64, log=quiet)
    sa = torch.load(tmp / 'r_full' / 'weights_e003.pt'); sb = torch.load(tmp / 'r_part' / 'weights_e003.pt')
    add('E08', 'a stopped-and-resumed run equals an uninterrupted run (weights and validation history)', all(torch.equal(sa[k], sb[k]) for k in sa) and [h['val_mae'] for h in d1['history']] == [h['val_mae'] for h in d2['history']])
    # E09 seed averaging and per-set summary by hand
    r = [{'loss': np.array([1., 2.], np.float32), 'chg_sum': np.array([2., 4.], np.float32), 'n_chg': np.array([1, 2], np.int16), 'spr': np.array([0., 1.], np.float32)}, {'loss': np.array([3., 4.], np.float32), 'chg_sum': np.array([4., 6.], np.float32), 'n_chg': np.array([1, 2], np.int16), 'spr': np.array([2., 3.], np.float32)}]
    av = average_results(r); add('E09', 'seeds are averaged per window before any statistic', np.allclose(av['loss'], [2, 3]) and np.allclose(av['chg_sum'], [3, 5]) and np.allclose(av['spr'], [1, 2]))
    # E10 sampler: deterministic per (seed, epoch), training bank holds only training sets, validation bank only validation
    p1 = np.random.default_rng([3, 2]).permutation(1000)[:100]; p2 = np.random.default_rng([3, 2]).permutation(1000)[:100]
    add('E10', 'epoch sample is deterministic per (seed, epoch); training bank = training sets only; validation bank = validation sets only', np.array_equal(p1, p2) and all(btr.summ[k]['split'] == 'train' for k in btr.keys) and all(bva.summ[k]['split'] == 'val' for k in bva.keys))
    # E11 frozen config
    try: check_frozen(tmp / 'fz.json', {'a': 1}); check_frozen(tmp / 'fz.json', {'a': 1}); refused = False; check_frozen(tmp / 'fz.json', {'a': 2})
    except RuntimeError: refused = True
    add('E11', 'a changed frozen config is refused', refused)
    # E12 put_json never overwrites
    put_json({'a': 1}, tmp / 'p.json'); put_json({'a': 2}, tmp / 'p.json'); add('E12', 'put_json keeps the first file and writes a changed one as .r1', json.loads((tmp / 'p.json').read_text()) == {'a': 1} and (tmp / 'p.r1.json').exists())
    # E13 B4 and tune_b2 run and give sane numbers
    tb = tune_b2(bva); sc_ = evaluate(bva, pred_b4, extra=b4_score); add('E13', 'B2 tuning and B4 run; B4 score is finite and non-negative', tb['span_node'] in SPANS and np.isfinite(sc_['extra']).all() and (sc_['extra'] >= 0).all())
    return out

Writing staged_lib.py


In [4]:
%%writefile staged_diag_lib.py
"""D1b diagnostic library: how much can ANY simple predictor beat persistence under the frozen MAE metric?
Validation/training windows only. Nothing here changes a frozen rule."""
import json, time, math
from pathlib import Path
import numpy as np
import torch, torch.nn as nn
import staged_lib as S
from staged_lib import SEQ, NN, NT, LOGQ, GLS, GIMB

NH = 8                                   # number of recent changes shown to the shared-node net
NCLS = 5                                 # change classes: neg-large, neg-small, zero, pos-small, pos-large
NCNT = 3                                 # number of changes in the last 5 intervals: 0, 1, 2+

# ---------------------------------------------------------------- per-window quantities (all in scaled units)
def window_deltas(xn, xg, y, last):
    """returns: dn_hist (B,NH,50) the last NH changes of log_qty per node, d_last (B,50), d_next (B,50), sp_hist (B,NH), sp_last (B,), sp_next (B,)"""
    q = xn[:, :, :, LOGQ]; dq = q[:, 1:] - q[:, :-1]                    # (B,49,50)
    s = xg[:, :, GLS]; ds = s[:, 1:] - s[:, :-1]                        # (B,49)
    d_next = y[:, :NN] - last[:, :NN]; sp_next = y[:, NN] - last[:, NN]
    return dq[:, -NH:], dq[:, -1], d_next, ds[:, -NH:], ds[:, -1], sp_next, dq[:, -5:], ds[:, -5:]
def change_class(d, edge):
    """0 neg-large, 1 neg-small, 2 zero, 3 pos-small, 4 pos-large; zero is EXACTLY zero (values are stored, not computed, so unchanged quantities are bit-identical)"""
    a = d.abs(); c = torch.full(d.shape, 2, dtype=torch.long, device=d.device)
    c = torch.where(d > 0, torch.where(a >= edge, torch.full_like(c, 4), torch.full_like(c, 3)), c)
    c = torch.where(d < 0, torch.where(a >= edge, torch.zeros_like(c), torch.ones_like(c)), c)
    return c
def count_class(d5):
    k = (d5 != 0).sum(1); return torch.clamp(k, max=NCNT - 1)

# ---------------------------------------------------------------- 1. per-node change statistics
@torch.no_grad()
def node_stats(bank, windows, bs=2048):
    z = lambda: torch.zeros(NN, dtype=torch.float64)
    A = {k: z() for k in ('n', 'chg', 'prev_chg', 'chg_given_prev', 'chg_given_noprev', 'both', 'reverse', 'abs_b1')}; sp = {'n': 0, 'chg': 0, 'abs_b1': 0.0}
    for a in range(0, len(windows), bs):
        g = torch.from_numpy(windows[a:a + bs]).to(bank.device); xn, xg, y, last = bank.batch(g)
        _, dl, dn, _, sl, sn, _, _ = window_deltas(xn, xg, y, last)
        ch = (dn != 0); pv = (dl != 0)
        A['n'] += torch.full((NN,), len(g), dtype=torch.float64); A['chg'] += ch.sum(0).double().cpu(); A['prev_chg'] += pv.sum(0).double().cpu()
        A['chg_given_prev'] += (ch & pv).sum(0).double().cpu(); A['chg_given_noprev'] += (ch & ~pv).sum(0).double().cpu()
        both = ch & pv; A['both'] += both.sum(0).double().cpu(); A['reverse'] += (both & (torch.sign(dn) == -torch.sign(dl))).sum(0).double().cpu(); A['abs_b1'] += dn.abs().sum(0).double().cpu()
        sp['n'] += len(g); sp['chg'] += int((sn != 0).sum()); sp['abs_b1'] += float(sn.abs().sum())
    n = A['n']; out = {'p_change': (A['chg'] / n).numpy(), 'p_change_given_prev': (A['chg_given_prev'] / A['prev_chg'].clamp(min=1)).numpy(),
                       'p_change_given_noprev': (A['chg_given_noprev'] / (n - A['prev_chg']).clamp(min=1)).numpy(), 'p_reverse_given_both': (A['reverse'] / A['both'].clamp(min=1)).numpy(),
                       'mae_b1': (A['abs_b1'] / n).numpy(), 'spread_p_change': sp['chg'] / sp['n'], 'spread_mae_b1': sp['abs_b1'] / sp['n']}
    return out
def rank_groups(v):
    """average the 25 bid and 25 ask ranks into rank buckets 1-3, 4-10, 11-25 (bid and ask averaged)"""
    v = np.asarray(v); r = {}
    for nm, (a, b) in {'rank 1-3': (0, 3), 'rank 4-10': (3, 10), 'rank 11-25': (10, 25)}.items(): r[nm] = float((v[a:b].mean() + v[25 + a:25 + b].mean()) / 2)
    return r

# ---------------------------------------------------------------- 2. binned conditional-median predictor (fit on TRAIN windows, scored on validation)
@torch.no_grad()
def fit_edge(bank, windows, bs=2048):
    vals = []
    for a in range(0, len(windows), bs):
        g = torch.from_numpy(windows[a:a + bs]).to(bank.device); xn, xg, y, last = bank.batch(g); dl = window_deltas(xn, xg, y, last)[1]; vals.append(dl.abs()[dl != 0].cpu())
    v = torch.cat(vals); return float(v.median()) if len(v) else 1.0
@torch.no_grad()
def _bins(xn, xg, y, last, edge):
    dh, dl, dn, sh, sl, sn, d5, s5 = window_deltas(xn, xg, y, last)
    cn = change_class(dl, edge) * NCNT + count_class(d5)[...]                     # (B,50)  class index inside a node
    cn = cn + torch.arange(NN, device=xn.device)[None, :] * (NCLS * NCNT)        # node-specific bin
    cs = change_class(sl[:, None], edge)[:, 0] * NCNT + count_class(s5[:, :, None])[:, 0]
    return cn, cs, dn, sn
@torch.no_grad()
def fit_binned(bank, windows, edge, bs=2048):
    nb = NN * NCLS * NCNT; node_vals = [[] for _ in range(nb)]; sp_vals = [[] for _ in range(NCLS * NCNT)]; allb, alld, sb, sd = [], [], [], []
    for a in range(0, len(windows), bs):
        g = torch.from_numpy(windows[a:a + bs]).to(bank.device); xn, xg, y, last = bank.batch(g); cn, cs, dn, sn = _bins(xn, xg, y, last, edge)
        allb.append(cn.flatten().cpu().numpy()); alld.append(dn.flatten().cpu().numpy()); sb.append(cs.cpu().numpy()); sd.append(sn.cpu().numpy())
    b = np.concatenate(allb); d = np.concatenate(alld); node_med = np.zeros(nb, np.float32); node_n = np.bincount(b, minlength=nb)
    order = np.argsort(b, kind='stable'); b_s = b[order]; d_s = d[order]; starts = np.searchsorted(b_s, np.arange(nb + 1))
    for i in range(nb):
        if starts[i + 1] > starts[i]: node_med[i] = np.median(d_s[starts[i]:starts[i + 1]])
    sbb = np.concatenate(sb); sdd = np.concatenate(sd); sp_med = np.zeros(NCLS * NCNT, np.float32)
    for i in range(NCLS * NCNT):
        m = sbb == i
        if m.any(): sp_med[i] = np.median(sdd[m])
    return {'node_med': node_med, 'spread_med': sp_med, 'edge': edge, 'node_n': node_n}
@torch.no_grad()
def eval_binned(bank, windows, model, bs=2048):
    nm = torch.from_numpy(model['node_med']).to(bank.device); sm = torch.from_numpy(model['spread_med']).to(bank.device); R = _acc()
    for a in range(0, len(windows), bs):
        g = torch.from_numpy(windows[a:a + bs]).to(bank.device); xn, xg, y, last = bank.batch(g); cn, cs, dn, sn = _bins(xn, xg, y, last, model['edge'])
        _add(R, nm[cn] - dn, sm[cs] - sn, dn, sn)
    return _fin(R)

# ---------------------------------------------------------------- accumulate errors (all-target MAE with equal weight per target value; changed-target MAE)
def _acc(): return {'abs_all': 0.0, 'n_all': 0, 'abs_chg': 0.0, 'n_chg': 0, 'abs_b1_all': 0.0, 'abs_b1_chg': 0.0, 'sp_abs': 0.0, 'sp_b1': 0.0, 'n_w': 0, 'rank': {}}
def _add(R, en, es, dn, sn):
    en = en.abs(); es = es.abs(); m = (dn != 0); ms = (sn != 0)
    R['abs_all'] += float(en.sum() + es.sum()); R['n_all'] += en.numel() + es.numel(); R['abs_b1_all'] += float(dn.abs().sum() + sn.abs().sum())
    R['abs_chg'] += float((en * m).sum() + (es * ms).sum()); R['n_chg'] += int(m.sum() + ms.sum()); R['abs_b1_chg'] += float((dn.abs() * m).sum() + (sn.abs() * ms).sum())
    R['sp_abs'] += float(es.sum()); R['sp_b1'] += float(sn.abs().sum()); R['n_w'] += len(sn)
    for nm, (a, b) in {'rank 1-3': (0, 3), 'rank 4-10': (3, 10), 'rank 11-25': (10, 25)}.items():
        idx = list(range(a, b)) + list(range(25 + a, 25 + b)); e = en[:, idx]; d = dn[:, idx].abs(); r = R['rank'].setdefault(nm, [0.0, 0.0, 0])
        r[0] += float(e.sum()); r[1] += float(d.sum()); r[2] += e.numel()
def _fin(R):
    out = {'mae_all': R['abs_all'] / R['n_all'], 'mae_all_b1': R['abs_b1_all'] / R['n_all'], 'mae_changed': R['abs_chg'] / max(R['n_chg'], 1), 'mae_changed_b1': R['abs_b1_chg'] / max(R['n_chg'], 1),
           'mae_spread': R['sp_abs'] / R['n_w'], 'mae_spread_b1': R['sp_b1'] / R['n_w'], 'n_windows': R['n_w']}
    out['rel_all_pct'] = 100 * (1 - out['mae_all'] / out['mae_all_b1']); out['rel_changed_pct'] = 100 * (1 - out['mae_changed'] / out['mae_changed_b1']); out['rel_spread_pct'] = 100 * (1 - out['mae_spread'] / max(out['mae_spread_b1'], 1e-12))
    out['by_rank'] = {k: {'mae': v[0] / v[2], 'mae_b1': v[1] / v[2], 'rel_pct': 100 * (1 - v[0] / max(v[1], 1e-12))} for k, v in R['rank'].items()}
    return out

# ---------------------------------------------------------------- 3. shared-node L1 network (one small net applied to every node; asks "is there any local information at all?")
class SharedNet(nn.Module):
    def __init__(s, h=128):
        super().__init__(); s.emb = nn.Embedding(NN, 8)
        s.node = nn.Sequential(nn.Linear(NH + 6 + 8 + 3 + NH, h), nn.ReLU(), nn.Linear(h, h), nn.ReLU(), nn.Linear(h, 2))
        s.spread = nn.Sequential(nn.Linear(NH + 3, 64), nn.ReLU(), nn.Linear(64, 1))
        nn.init.zeros_(s.node[-1].weight); nn.init.zeros_(s.node[-1].bias); nn.init.zeros_(s.spread[-1].weight); nn.init.zeros_(s.spread[-1].bias)
    def forward(s, xn, xg, y, last):
        dh, dl, dn, sh, sl, sn, _, _ = window_deltas(xn, xg, y, last); B = xn.shape[0]; cl = lambda v: torch.clamp(v, -10, 10)
        node_now = xn[:, -1]                                                      # (B,50,6) channels at t
        x = torch.cat([cl(dh).permute(0, 2, 1), node_now, s.emb.weight[None].expand(B, -1, -1), xg[:, -1][:, None, :].expand(-1, NN, -1), cl(sh)[:, None, :].expand(-1, NN, -1)], 2)
        o = s.node(x); dpred = o[..., 0]; logit = o[..., 1]
        sp = s.spread(torch.cat([cl(sh), xg[:, -1]], 1))[:, 0]
        return dpred, logit, sp, dn, sn
def train_shared(btr, bva, frac=0.10, epochs=3, bs=512, device='cpu', seed=0, log=print, lr=1e-3):
    torch.manual_seed(seed); net = SharedNet().to(device); opt = torch.optim.Adam(net.parameters(), lr=lr); n = len(btr.win); hist = []
    bce = nn.BCEWithLogitsLoss()
    for e in range(1, epochs + 1):
        t0 = time.time(); rng = np.random.default_rng([seed, e]); pick = rng.permutation(n)[:int(n * frac)]; net.train(); tot = 0.0; cnt = 0
        for a in range(0, len(pick), bs):
            g = torch.from_numpy(btr.win[pick[a:a + bs]]).to(device); xn, xg, y, last = btr.batch(g); dp, lg, sp, dn, sn = net(xn, xg, y, last)
            loss = (dp - dn).abs().mean() + (sp - sn).abs().mean() / NN + 0.2 * bce(lg, (dn != 0).float()); opt.zero_grad(set_to_none=True); loss.backward(); torch.nn.utils.clip_grad_norm_(net.parameters(), 1.0); opt.step(); tot += float(loss.detach()) * len(g); cnt += len(g)
        hist.append({'epoch': e, 'train_loss': tot / max(cnt, 1), 'seconds': round(time.time() - t0, 1)}); log(f'  shared net epoch {e}: loss {hist[-1]["train_loss"]:.5f} | {hist[-1]["seconds"]} s')
    return net, hist
@torch.no_grad()
def eval_shared(net, bank, windows, bs=1024, auc_max=1_500_000, seed=0):
    net.eval(); R = _acc(); rng = np.random.default_rng(seed); scores = []; labels = []; kept = 0
    for a in range(0, len(windows), bs):
        g = torch.from_numpy(windows[a:a + bs]).to(bank.device); xn, xg, y, last = bank.batch(g); dp, lg, sp, dn, sn = net(xn, xg, y, last)
        _add(R, dp - dn, sp - sn, dn, sn)
        if kept < auc_max:
            s_ = lg.flatten().cpu().numpy(); l_ = (dn != 0).flatten().cpu().numpy(); scores.append(s_); labels.append(l_); kept += len(s_)
    out = _fin(R); out['auc_change'] = auc(np.concatenate(labels), np.concatenate(scores)); out['base_rate_change'] = float(np.concatenate(labels).mean()); return out
def auc(labels, scores):
    """rank-based AUC (Mann-Whitney) with average ranks for ties"""
    labels = np.asarray(labels).astype(bool); scores = np.asarray(scores, dtype=np.float64); n1 = labels.sum(); n0 = len(labels) - n1
    if n1 == 0 or n0 == 0: return float('nan')
    order = np.argsort(scores, kind='mergesort'); s = scores[order]; ranks = np.empty(len(s)); i = 0
    while i < len(s):
        j = i
        while j + 1 < len(s) and s[j + 1] == s[i]: j += 1
        ranks[i:j + 1] = (i + j) / 2 + 1; i = j + 1
    r = np.empty(len(s)); r[order] = ranks; return float((r[labels].sum() - n1 * (n1 + 1) / 2) / (n1 * n0))
@torch.no_grad()
def eval_b1_reference(bank, windows, bs=2048):
    R = _acc()
    for a in range(0, len(windows), bs):
        g = torch.from_numpy(windows[a:a + bs]).to(bank.device); xn, xg, y, last = bank.batch(g); _, _, dn, _, _, sn, _, _ = window_deltas(xn, xg, y, last); _add(R, dn, sn, dn, sn)
    return _fin(R)

# ---------------------------------------------------------------- tests on synthetic data (can the diagnostic actually SEE a signal when one exists?)
def make_signal_bank(tmp, kind, n_int=3000, seed=0):
    """a Stage-C-shaped folder where log_qty follows a known rule: 'none' = random sparse jumps (nothing to learn), 'reverse' = every change is undone on the next interval (strong, learnable)"""
    root = S.make_mock_stagec(tmp, n_int=n_int, seed=seed); rng = np.random.default_rng(seed + 1)
    for split in ('train', 'val'):
        for day in S.SPLITS[split]:
            for sym in S.SYMS:
                d = Path(root) / 'sets' / f'{sym}_{day}'; nodes = np.load(d / 'nodes.npy').astype(np.float32); q = np.zeros((n_int, NN), np.float32); cur = rng.normal(size=NN).astype(np.float32); prev = cur.copy()
                for i in range(n_int):
                    if kind == 'reverse':
                        if i % 2 == 0: q[i] = cur
                        else: q[i] = cur + 1.0       # a bump at odd steps
                        # odd steps are +1 above the base, even steps return to the base: the change at odd t is +1 and is undone at t+1
                    else:
                        ch = rng.random(NN) < 0.2; cur = np.where(ch, cur + rng.normal(scale=0.5, size=NN), cur); q[i] = cur
                nodes[:, :, LOGQ] = q; np.save(d / 'nodes.npy', nodes.astype(np.float16))
                sm = json.loads((d / 'set_summary.json').read_text()); sm['hashes']['nodes.npy'] = S.sha256_file(d / 'nodes.npy'); S.dump_json(sm, d / 'set_summary.json')
    return root
def unit_tests(tmp):
    tmp = Path(tmp); out = []
    def add(i, nm, ok, det=''): out.append((i, nm, bool(ok), str(det)))
    r_none = make_signal_bank(tmp / 'none', 'none'); r_rev = make_signal_bank(tmp / 'rev', 'reverse')
    btr = S.Bank(r_none, S.keys_of('train')); bva = S.Bank(r_none, S.keys_of('val'))
    # F01 deltas and classes by hand
    xn = torch.zeros(1, SEQ, NN, S.NCH); xn[0, -1, 3, LOGQ] = 2.0; xn[0, -2, 3, LOGQ] = 0.5; xg = torch.zeros(1, SEQ, S.GCH); y = torch.zeros(1, NT); last = torch.zeros(1, NT)
    dh, dl, dn, sh, sl, sn, d5, s5 = window_deltas(xn, xg, y, last); cls = change_class(dl, 1.0)
    add('F01', 'last change, class and count of changes by hand', abs(float(dl[0, 3]) - 1.5) < 1e-6 and int(cls[0, 3]) == 4 and int(cls[0, 4]) == 2 and int(count_class(d5)[0, 3]) == 2, f'{float(dl[0,3])} class {int(cls[0,3])}')
    # F02 on pure noise nothing can beat persistence by much (binned and shared)
    wtr = btr.win[:6000]; wva = bva.win[:3000]; edge = fit_edge(btr, wtr); mb = fit_binned(btr, wtr, edge); eb = eval_binned(bva, wva, mb)
    net, _ = train_shared(btr, bva, frac=0.5, epochs=2, bs=128, log=lambda *a, **k: None); es = eval_shared(net, bva, wva)
    add('F02', 'no-signal data: binned and shared-net improvement over persistence stays below 3%', eb['rel_all_pct'] < 3 and es['rel_all_pct'] < 3, f"binned {eb['rel_all_pct']:.2f}% shared {es['rel_all_pct']:.2f}%")
    # F03 strong-signal data: both diagnostics must see it
    ttr = S.Bank(r_rev, S.keys_of('train')); tva = S.Bank(r_rev, S.keys_of('val')); wtr2 = ttr.win[:6000]; wva2 = tva.win[:3000]; e2 = fit_edge(ttr, wtr2); m2 = fit_binned(ttr, wtr2, e2); b2 = eval_binned(tva, wva2, m2)
    n2, _ = train_shared(ttr, tva, frac=0.5, epochs=3, bs=128, log=lambda *a, **k: None, lr=3e-3); s2 = eval_shared(n2, tva, wva2)
    add('F03', 'strong-signal data (every change is undone next step): binned improves a lot, shared net improves clearly', b2['rel_all_pct'] > 30 and s2['rel_all_pct'] > 5, f"binned {b2['rel_all_pct']:.1f}% shared {s2['rel_all_pct']:.1f}%")
    # F04 AUC function
    a1 = auc([0, 0, 1, 1], [0.1, 0.2, 0.8, 0.9]); a2 = auc([0, 1, 0, 1], [0.5, 0.5, 0.5, 0.5]); rng = np.random.default_rng(0); a3 = auc(rng.integers(0, 2, 20000), rng.normal(size=20000))
    add('F04', 'AUC: perfect = 1, all tied = 0.5, random about 0.5', a1 == 1.0 and abs(a2 - 0.5) < 1e-9 and abs(a3 - 0.5) < 0.02)
    # F05 statistics: persistence reference equals node_stats mae_b1
    ns = node_stats(btr, wtr); ref = eval_b1_reference(btr, wtr); add('F05', 'persistence error from two code paths agrees', abs(ref['mae_all_b1'] - (ns['mae_b1'].sum() + ns['spread_mae_b1']) / NT) < 1e-6)
    return out

Writing staged_diag_lib.py


In [5]:
%%writefile staged_d2_lib.py
"""D2 library: TGNN M1 (rank-chain + best-bid/best-ask + global edges), M0 (self + global links only), B3a (MLP with the same auxiliary loss),
resumable trainer with a time budget, change-occurrence AUC. Uses the frozen D1 library unchanged."""
import os, json, time
from pathlib import Path
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
import staged_lib as S
from staged_lib import SEQ, NN, NCH, GCH, NT, LOGQ, GLS
NG = NN + 1                                   # 50 level nodes + 1 global node

# ---------------------------------------------------------------- graph
def build_adj():
    """row-normalised adjacency per edge type. Node order: bid rank 1..25 (0..24), ask rank 1..25 (25..49), global (50)."""
    chain = np.zeros((NG, NG), np.float32); spread = np.zeros((NG, NG), np.float32); glob = np.zeros((NG, NG), np.float32)
    for r in range(24):
        chain[r, r + 1] = chain[r + 1, r] = 1; chain[25 + r, 26 + r] = chain[26 + r, 25 + r] = 1
    spread[0, 25] = spread[25, 0] = 1
    glob[:NN, NN] = 1; glob[NN, :NN] = 1
    out = {}
    for k, a in (('chain', chain), ('spread', spread), ('glob', glob)):
        rs = a.sum(1, keepdims=True); out[k] = a / np.maximum(rs, 1)
    return out, {'chain': chain, 'spread': spread, 'glob': glob}
EDGES_M1 = ('chain', 'spread', 'glob'); EDGES_M0 = ('glob',)

class TGNN(nn.Module):
    def __init__(s, h=64, edges=EDGES_M1, layers=2):
        super().__init__(); s.h = h; s.edges = tuple(edges); s.layers = layers; adj, _ = build_adj()
        s.emb = nn.Embedding(NG, 8); s.enc_n = nn.Linear(NCH + 8, h); s.enc_g = nn.Linear(GCH + 8, h)
        s.self_lin = nn.ModuleList([nn.Linear(h, h) for _ in range(layers)]); s.norm = nn.ModuleList([nn.LayerNorm(h) for _ in range(layers)])
        s.edge_lin = nn.ModuleList([nn.ModuleDict({e: nn.Linear(h, h, bias=False) for e in s.edges}) for _ in range(layers)])
        for e in ('chain', 'spread', 'glob'): s.register_buffer('A_' + e, torch.from_numpy(adj[e]))
        s.gru = nn.GRU(h, h, batch_first=True)
        s.head_n = nn.Sequential(nn.Linear(2 * h, h), nn.ReLU(), nn.Linear(h, 2)); s.head_g = nn.Sequential(nn.Linear(2 * h, h), nn.ReLU(), nn.Linear(h, 1))
        for m in (s.head_n[-1], s.head_g[-1]): nn.init.zeros_(m.weight); nn.init.zeros_(m.bias)         # starts exactly at persistence
    def forward(s, xn, xg, last):
        B, T = xn.shape[0], xn.shape[1]; em = s.emb.weight
        n_in = torch.cat([xn, em[:NN][None, None].expand(B, T, -1, -1)], -1); g_in = torch.cat([xg[:, :, None, :], em[NN][None, None, None].expand(B, T, 1, -1)], -1)
        h = torch.cat([s.enc_n(n_in), s.enc_g(g_in)], 2)                                              # (B,T,51,h)
        for l in range(s.layers):
            agg = s.self_lin[l](h)
            for e in s.edges: agg = agg + s.edge_lin[l][e](torch.matmul(getattr(s, 'A_' + e).to(h.dtype), h))
            h = h + F.relu(s.norm[l](agg))
        seq = h.permute(0, 2, 1, 3).reshape(B * NG, T, s.h); out, _ = s.gru(seq)
        feat = torch.cat([out[:, -1], h[:, -1].reshape(B * NG, s.h)], 1).reshape(B, NG, 2 * s.h)
        o = s.head_n(feat[:, :NN]); dn = o[..., 0]; logit = o[..., 1]; dsp = s.head_g(feat[:, NN])[:, 0]
        return last + torch.cat([dn, dsp[:, None]], 1).float(), logit.float()
def make_m1(cfg): return TGNN(cfg['hidden'], EDGES_M1)
def make_m0(cfg): return TGNN(cfg['hidden'], EDGES_M0)

class MLPA(nn.Module):
    """B3a: the D1 MLP (same sizes) with an extra change-occurrence output, trained with the same loss as M0 and M1"""
    def __init__(s, h):
        super().__init__(); s.net = nn.Sequential(nn.Linear(SEQ * NN * NCH + SEQ * GCH, h), nn.ReLU(), nn.Linear(h, h), nn.ReLU(), nn.Linear(h, NT + NN))
        nn.init.zeros_(s.net[-1].weight); nn.init.zeros_(s.net[-1].bias)
    def forward(s, xn, xg, last):
        o = s.net(torch.cat([xn.flatten(1), xg.flatten(1)], 1)).float(); return last + o[:, :NT], o[:, NT:]
def make_mlpa(cfg): return MLPA(cfg['hidden'])

# ---------------------------------------------------------------- loss, validation helpers
def loss_fn(model_out, y, last, aux):
    pred, logit = model_out; l1 = (pred - y).abs().mean(); bce = F.binary_cross_entropy_with_logits(logit, (y[:, :NN] != last[:, :NN]).float()); return l1 + aux * bce, l1, bce
def amp_ctx(device, amp): return torch.autocast('cuda', dtype=torch.float16, enabled=bool(amp and device == 'cuda'))
@torch.no_grad()
def val_mae(bank, windows, model, device, amp, bs=512):
    model.eval(); tot = 0.0; n = 0
    for a in range(0, len(windows), bs):
        g = torch.from_numpy(windows[a:a + bs]).to(device); xn, xg, y, last = bank.batch(g)
        with amp_ctx(device, amp): pred, _ = model(xn, xg, last)
        tot += float((pred.float() - y).abs().mean(1).sum()); n += len(g)
    return tot / max(n, 1)
def val_subset(bank, frac=0.25, seed=12345):
    """fixed random share of the validation windows of every set (so both symbols are represented); sorted for reproducibility"""
    rng = np.random.default_rng(seed); parts = []
    for i in range(len(bank.keys)):
        w = bank.win[bank.win_set == i]; parts.append(np.sort(rng.choice(w, int(len(w) * frac), replace=False)))
    return np.concatenate(parts)
@torch.no_grad()
def change_auc(model, bank, windows, device, amp, bs=512, max_pairs=1_500_000):
    model.eval(); sc = []; lb = []; kept = 0
    for a in range(0, len(windows), bs):
        g = torch.from_numpy(windows[a:a + bs]).to(device); xn, xg, y, last = bank.batch(g)
        with amp_ctx(device, amp): _, logit = model(xn, xg, last)
        sc.append(logit.float().flatten().cpu().numpy()); lb.append((y[:, :NN] != last[:, :NN]).flatten().cpu().numpy()); kept += sc[-1].size
        if kept >= max_pairs: break
    s_ = np.concatenate(sc); l_ = np.concatenate(lb); p = 1 / (1 + np.exp(-np.clip(s_, -30, 30))); bce = float(-(l_ * np.log(p + 1e-9) + (~l_) * np.log(1 - p + 1e-9)).mean())
    from staged_diag_lib import auc
    return {'auc': auc(l_, s_), 'bce': bce, 'base_rate': float(l_.mean()), 'pairs': int(len(l_))}
def predict_fn(model, device, amp):
    def f(xn, xg, last):
        with amp_ctx(device, amp): return model(xn, xg, last)[0].float()
    return f

# ---------------------------------------------------------------- resumable trainer with a time budget
def fit2(make_model, cfg, bank_tr, bank_va, val_windows, out_dir, seed, aux=0.2, frac=0.10, max_epochs=20, patience=3, bs=128, device='cpu', amp=False, deadline=None, log=print, _halt_after=None):
    out = Path(out_dir); out.mkdir(parents=True, exist_ok=True)
    if (out / 'done.json').exists(): return json.loads((out / 'done.json').read_text())
    assert all(bank_tr.summ[k]['split'] == 'train' for k in bank_tr.keys), 'training bank contains a non-training set'
    torch.manual_seed(seed); model = make_model(cfg).to(device); opt = torch.optim.Adam(model.parameters(), lr=cfg['lr']); scaler = torch.amp.GradScaler('cuda', enabled=bool(amp and device == 'cuda'))
    state = {'epoch': 0, 'best': float('inf'), 'best_epoch': 0, 'bad': 0, 'hist': []}
    for p in sorted(out.glob('resume_e*.pt'), reverse=True):
        try: ck = torch.load(p, map_location='cpu', weights_only=False)
        except Exception: continue
        model.load_state_dict(ck['model']); opt.load_state_dict(ck['opt']); scaler.load_state_dict(ck['scaler']); torch.set_rng_state(ck['rng_cpu'])
        if ck.get('rng_cuda') is not None and torch.cuda.is_available(): torch.cuda.set_rng_state_all(ck['rng_cuda'])
        state = ck['state']; log(f'  resumed from {p.name} (epoch {state["epoch"]})'); break
    n = len(bank_tr.win)
    while state['epoch'] < max_epochs and state['bad'] < patience:
        if deadline is not None and time.time() > deadline: log('  time budget reached before this epoch: stop here, run again to continue'); return None
        e = state['epoch'] + 1; t0 = time.time(); rng = np.random.default_rng([seed, e]); pick = rng.permutation(n)[:int(n * frac)]
        model.train(); tot = torch.zeros((), device=device); cnt = 0
        for a in range(0, len(pick), bs):
            g = torch.from_numpy(bank_tr.win[pick[a:a + bs]]).to(device); xn, xg, y, last = bank_tr.batch(g)
            with amp_ctx(device, amp): out_ = model(xn, xg, last)
            loss, l1, bce = loss_fn(out_, y, last, aux); opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); scaler.step(opt); scaler.update()
            tot += l1.detach() * len(g); cnt += len(g)
        t_train = time.time() - t0; val = val_mae(bank_va, val_windows, model, device, amp); torch.save(model.state_dict(), out / f'weights_e{e:03d}.pt')
        state['epoch'] = e; state['hist'].append({'epoch': e, 'train_l1': float(tot / max(cnt, 1)), 'val_mae_sub': val, 'train_seconds': round(t_train, 1), 'epoch_seconds': round(time.time() - t0, 1)})
        if val < state['best'] - 1e-9: state['best'] = val; state['best_epoch'] = e; state['bad'] = 0
        else: state['bad'] += 1
        tmp = out / f'resume_e{e:03d}.pt.tmp'
        torch.save({'model': model.state_dict(), 'opt': opt.state_dict(), 'scaler': scaler.state_dict(), 'rng_cpu': torch.get_rng_state(), 'rng_cuda': torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None, 'state': state}, tmp); os.replace(tmp, out / f'resume_e{e:03d}.pt')
        for p in out.glob('resume_e*.pt'):
            if int(p.stem[8:]) < e - 1: p.unlink()
        with open(out / 'log.jsonl', 'a') as f: f.write(json.dumps(state['hist'][-1]) + '\n')
        log(f'  epoch {e}: train L1 {state["hist"][-1]["train_l1"]:.5f} | val(sub) {val:.5f} | best {state["best"]:.5f} (e{state["best_epoch"]}) | {state["hist"][-1]["epoch_seconds"]} s')
        if _halt_after is not None and e >= _halt_after: return None
    done = {'cfg': cfg, 'seed': seed, 'best_val_sub': state['best'], 'best_epoch': state['best_epoch'], 'epochs_run': state['epoch'], 'history': state['hist'], 'n_params': int(sum(p.numel() for p in model.parameters())),
            'mean_epoch_seconds': float(np.mean([h['epoch_seconds'] for h in state['hist']])), 'frac': frac, 'max_epochs': max_epochs, 'patience': patience, 'batch_size': bs, 'aux_weight': aux, 'amp': bool(amp and device == 'cuda')}
    S.dump_json(done, out / 'done.json'); return done
def load_best2(make_model, cfg, out_dir, device='cpu'):
    d = json.loads((Path(out_dir) / 'done.json').read_text()); m = make_model(cfg).to(device); m.load_state_dict(torch.load(Path(out_dir) / f'weights_e{d["best_epoch"]:03d}.pt', map_location=device)); m.eval(); return m, d

# ---------------------------------------------------------------- tests
def unit_tests(tmp):
    import staged_diag_lib as D
    tmp = Path(tmp); out = []
    def add(i, nm, ok, det=''): out.append((i, nm, bool(ok), str(det)))
    adj, raw = build_adj(); n_chain = int(raw['chain'].sum() // 2); n_glob = int(raw['glob'].sum() // 2)
    add('G01', 'graph: 48 chain edges (24 per side), 1 best-bid/best-ask edge joining nodes 0 and 25, global linked to all 50 nodes, rows normalised', n_chain == 48 and raw['spread'][0, 25] == 1 and raw['spread'].sum() == 2 and n_glob == 50 and np.allclose(adj['glob'][:NN].sum(1), 1) and raw['chain'][24, 25] == 0, f'chain {n_chain}, global {n_glob}')
    root = D.make_signal_bank(tmp / 'sc', 'reverse', n_int=2600); btr = S.Bank(root, S.keys_of('train')); bva = S.Bank(root, S.keys_of('val'))
    xn, xg, y, last = btr.batch(torch.tensor(btr.win[:6])); torch.manual_seed(0); m1 = TGNN(16, EDGES_M1); m0 = TGNN(16, EDGES_M0)
    add('G02', 'M1 and M0 start exactly at persistence', torch.equal(m1(xn, xg, last)[0], last) and torch.equal(m0(xn, xg, last)[0], last))
    # G03: M1 with chain and spread weights set to zero equals M0 (the only difference is the graph structure)
    torch.manual_seed(1); a = TGNN(16, EDGES_M1); b = TGNN(16, EDGES_M0); sd = a.state_dict()
    for k in list(sd):
        if 'edge_lin' in k and ('chain' in k or 'spread' in k): sd[k] = torch.zeros_like(sd[k])
    a.load_state_dict(sd); bs_ = {k: v for k, v in sd.items() if not ('edge_lin' in k and ('chain' in k or 'spread' in k))}; b.load_state_dict(bs_, strict=True)
    for m in (a, b):
        for hd in (m.head_n[-1], m.head_g[-1]): nn.init.normal_(hd.weight, std=0.1)
    b.head_n[-1].load_state_dict(a.head_n[-1].state_dict()); b.head_g[-1].load_state_dict(a.head_g[-1].state_dict())
    add('G03', 'M1 with its chain and best-bid/best-ask weights set to zero gives the same output as M0', torch.allclose(a(xn, xg, last)[0], b(xn, xg, last)[0], atol=1e-5))
    # G04: with one spatial layer, a neighbour's input reaches a node in M1 but not in M0 (M0 only has global links)
    torch.manual_seed(2); p1 = TGNN(16, EDGES_M1, layers=1); p0 = TGNN(16, EDGES_M0, layers=1)
    for m in (p1, p0):
        for hd in (m.head_n[-1], m.head_g[-1]): nn.init.normal_(hd.weight, std=0.1)
    x2 = xn.clone(); x2[:, :, 6, :] += 1.0; d1 = (p1(xn, xg, last)[0][:, 5] - p1(x2, xg, last)[0][:, 5]).abs().max(); d0 = (p0(xn, xg, last)[0][:, 5] - p0(x2, xg, last)[0][:, 5]).abs().max()
    add('G04', 'one spatial layer: node 6 input changes node 5 output in M1 (chain edge) but not in M0', float(d1) > 1e-6 and float(d0) < 1e-7, f'M1 {float(d1):.2e} M0 {float(d0):.2e}')
    # G05: loss pieces by hand
    pred = torch.zeros(1, NT); yy = torch.zeros(1, NT); yy[0, 3] = 1; ll = torch.zeros(1, NT); lg = torch.zeros(1, NN); l_, l1, bce = loss_fn((pred, lg), yy, ll, 0.2)
    add('G05', 'loss = L1 + 0.2 x BCE with the change mask as label', abs(float(l1) - 1 / NT) < 1e-6 and abs(float(bce) - np.log(2)) < 1e-6 and abs(float(l_) - (1 / NT + 0.2 * np.log(2))) < 1e-6)
    # G06: stopped-and-resumed run equals uninterrupted run; time budget stop returns None and keeps files
    cfg = {'hidden': 8, 'lr': 2e-3}; quiet = lambda *a, **k: None; vs = val_subset(bva, 0.5)
    d1_ = fit2(make_m1, cfg, btr, bva, vs, tmp / 'full', seed=3, frac=0.15, max_epochs=3, patience=9, bs=64, log=quiet)
    fit2(make_m1, cfg, btr, bva, vs, tmp / 'part', seed=3, frac=0.15, max_epochs=3, patience=9, bs=64, log=quiet, _halt_after=1)
    r = fit2(make_m1, cfg, btr, bva, vs, tmp / 'dead', seed=3, frac=0.15, max_epochs=3, patience=9, bs=64, log=quiet, deadline=time.time() - 1)
    d2_ = fit2(make_m1, cfg, btr, bva, vs, tmp / 'part', seed=3, frac=0.15, max_epochs=3, patience=9, bs=64, log=quiet)
    sa = torch.load(tmp / 'full' / 'weights_e003.pt'); sb = torch.load(tmp / 'part' / 'weights_e003.pt')
    add('G06', 'resume equals an uninterrupted run; an exhausted time budget stops cleanly (returns None)', r is None and all(torch.equal(sa[k], sb[k]) for k in sa) and [h['val_mae_sub'] for h in d1_['history']] == [h['val_mae_sub'] for h in d2_['history']])
    # G07: the TGNN can learn a real signal (every change is undone): validation MAE falls clearly below persistence; M1 and B3a run through the same trainer
    cfg2 = {'hidden': 16, 'lr': 3e-3}; dm = fit2(make_m1, cfg2, btr, bva, vs, tmp / 'learn', seed=5, frac=0.4, max_epochs=4, patience=9, bs=64, log=quiet); base = float(S.evaluate(bva, S.pred_b1)['loss'].mean())
    da = fit2(make_mlpa, {'hidden': 16, 'lr': 3e-3}, btr, bva, vs, tmp / 'mlpa', seed=5, frac=0.4, max_epochs=2, patience=9, bs=64, log=quiet)
    add('G07', 'M1 beats persistence by more than 10% when the data hold a learnable signal; B3a runs with the same trainer', dm['best_val_sub'] < 0.9 * base and da['epochs_run'] == 2, f"M1 {dm['best_val_sub']:.4f} vs persistence {base:.4f}")
    # G08: validation subset: fixed, covers both symbols, only validation windows
    v1 = val_subset(bva, 0.25); v2 = val_subset(bva, 0.25); sets = {int(bva.win_set[np.searchsorted(bva.win, w)]) for w in v1[:50]} | {int(bva.win_set[np.searchsorted(bva.win, w)]) for w in v1[-50:]}
    add('G08', 'validation subset is deterministic, sorted, inside the validation bank and covers both symbols', np.array_equal(v1, v2) and np.isin(v1, bva.win).all() and len(sets) == 2, f'{len(v1)} of {len(bva.win)}')
    # G09: change AUC works on the learned model
    m, _ = load_best2(make_m1, cfg2, tmp / 'learn'); rn = D.make_signal_bank(tmp / 'nonebank', 'none', n_int=1500); bn = S.Bank(rn, S.keys_of('val')); ca = change_auc(m, bn, bn.win[:1500], 'cpu', False); add('G09', 'change-occurrence AUC runs and is finite on a trained model', np.isfinite(ca['auc']) and 0 <= ca['auc'] <= 1, ca)
    return out

Writing staged_d2_lib.py


## 3 · Safety checks: device, Stage C, finished D1, library hash equal to D1, frozen configuration

In [6]:
import torch
sys.path.insert(0, str(Path.cwd())); import staged_lib as S, staged_diag_lib as D, staged_d2_lib as L
LIB_SHA = S.sha256_file(Path('staged_lib.py')); DLIB_SHA = S.sha256_file(Path('staged_diag_lib.py')); D2LIB_SHA = S.sha256_file(Path('staged_d2_lib.py'))
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'; GPU = torch.cuda.get_device_name(0) if DEV == 'cuda' else None
SC = BASE / STAGEC_NAME; assert (SC / 'sets').is_dir(), f'Stage C folder not found: {SC}'
sc_file = json.loads((SC / 'config' / 'scaler_train_only.json').read_text()); SC_SHA = sc_file['sha256']
D1 = BASE / D1_NAME; rep1 = json.loads((D1 / 'reports' / 'd1_report.json').read_text()); assert rep1['verdict'] == 'D1 COMPLETE', 'D1 is not complete'
assert rep1['library_sha256'][:12] == LIB_SHA[:12], f'staged_lib.py differs from the one used in D1 ({rep1["library_sha256"][:12]} vs {LIB_SHA[:12]})'
B1_NPZ = D1 / 'd1' / 'baselines' / 'b1_val.npz'; B3_NPZ = D1 / 'd1' / 'b3' / 'b3_val_seed_averaged.npz'; B3_TUNE = D1 / 'd1' / 'b3' / 'tuning.json'
for p in (B1_NPZ, B3_NPZ, B3_TUNE): assert p.exists(), f'D1 result missing: {p}'
B3_CFG = json.loads(B3_TUNE.read_text())['best_config']
D1B_REP = json.loads((D1 / 'd1b' / 'd1b_report.json').read_text()) if (D1 / 'd1b' / 'd1b_report.json').exists() else None
OUT = D1 / 'd2'; (OUT / 'config').mkdir(parents=True, exist_ok=True); (OUT / 'runs').mkdir(exist_ok=True)
HW = {'device': DEV, 'gpu': GPU, 'torch': torch.__version__, 'python': platform.python_version(), 'drive_free_gb': round(shutil.disk_usage(str(BASE)).free / 1e9, 2)}
if DEV == 'cuda': f_, t_ = torch.cuda.mem_get_info(); HW['gpu_mem_total_gb'] = round(t_ / 1e9, 2); HW['gpu_mem_free_gb'] = round(f_ / 1e9, 2)
FROZEN = {'stagec': STAGEC_NAME, 'scaler_sha256': SC_SHA, 'd1': D1_NAME, 'frac': FRAC, 'max_epochs': MAX_EPOCHS, 'patience': PATIENCE, 'batch': BATCH, 'aux': AUX, 'val_frac': VAL_FRAC, 'amp': AMP, 'seed_tune': SEED_TUNE, 'seeds': SEEDS, 'grid': GRID, 'b3_cfg': B3_CFG,
          'auc_windows': AUC_WINDOWS, 'library_sha256': LIB_SHA, 'diag_library_sha256': DLIB_SHA, 'd2_library_sha256': D2LIB_SHA, 'metric': 'MAE, equal weight per target value (unchanged)'}
print(json.dumps(HW)); print('frozen config:', S.check_frozen(OUT / 'config' / 'frozen_config_d2_r2.json', FROZEN), '| D2 folder:', OUT.relative_to(BASE), '| B3 configuration from D1:', B3_CFG, '| D1b report found:', D1B_REP is not None)
FREEZE_FILE = D1 / 'config' / 'freeze_d3.json'; assert not FREEZE_FILE.exists(), 'a freeze file exists: D2 must run before the test days are opened'

{"device": "cuda", "gpu": "Tesla T4", "torch": "2.11.0+cu130", "python": "3.13.15", "drive_free_gb": 15.18, "gpu_mem_total_gb": 15.64, "gpu_mem_free_gb": 15.53}
frozen config: matches | D2 folder: TGNN_StageD_v1/d2 | B3 configuration from D1: {'hidden': 64, 'lr': 0.0003} | D1b report found: True


## 4 · Tests: D1 library tests + D2 tests (graph structure, start at persistence, M1 equals M0 when the neighbour weights are zero, neighbour dependence only in M1, loss by hand, resume equals uninterrupted run, signal learned, deterministic validation subset)

In [7]:
import tempfile
t0 = time.time()
with tempfile.TemporaryDirectory() as t_: ER1 = S.unit_tests(t_)
with tempfile.TemporaryDirectory() as t_: ER2 = L.unit_tests(t_)
ER = ER1 + ER2; display(pd.DataFrame(ER, columns=['id', 'test', 'pass', 'detail']))
assert all(r[2] for r in ER), 'A test failed: stop here and send me this table.'
print(f'tests: {sum(r[2] for r in ER)}/{len(ER)} passed in {time.time()-t0:.0f} s')

,id,test,pass,detail
0,E01,test days refused until the freeze file exists,True,refused=True allowed_after=True
1,E02,"input = intervals t-49..t, target = interval t...",True,
2,E03,changing t+1 and t+2 does not change the input...,True,
3,E04,windows contain no flagged interval and stay i...,True,
4,E05,B1 equals a hand computation; B2 with span 1 e...,True,
5,E06,changed-target mask and per-window sums,True,
6,E07,B3 starts at persistence and its loss falls on...,True,0.0820->0.0329
7,E08,a stopped-and-resumed run equals an uninterrup...,True,
8,E09,seeds are averaged per window before any stati...,True,
9,E10,"epoch sample is deterministic per (seed, epoch...",True,


tests: 22/22 passed in 219 s


## 5 · Load training and validation tensors (test days stay closed) and align with the D1 result files

In [8]:
t0 = time.time(); BTR = S.Bank(SC, S.keys_of('train'), device=DEV, freeze_file=FREEZE_FILE); BVA = S.Bank(SC, S.keys_of('val'), device=DEV, freeze_file=FREEZE_FILE)
try: S.Bank(SC, S.keys_of('test')[:1], device='cpu', freeze_file=FREEZE_FILE); TEST_REFUSED = False
except PermissionError: TEST_REFUSED = True
assert TEST_REFUSED
VALSUB = L.val_subset(BVA, VAL_FRAC)
rng = np.random.default_rng(777); AUCW = np.sort(rng.choice(BVA.win, min(AUC_WINDOWS, len(BVA.win)), replace=False))
r1 = np.load(B1_NPZ); r3 = np.load(B3_NPZ); ALIGN = bool(np.array_equal(r1['win'], BVA.win) and np.array_equal(r3['win'], BVA.win) and np.array_equal(r1['win_set'], BVA.win_set))
assert ALIGN, 'validation windows differ from the D1 result files'
R_B1 = {k: r1[k] for k in ('loss', 'chg_sum', 'n_chg', 'spr')}; R_B3 = {k: r3[k] for k in ('loss', 'chg_sum', 'n_chg', 'spr')}
print(f'train {len(BTR.win):,} windows | validation {len(BVA.win):,} windows (early-stopping subset {len(VALSUB):,}) | test days refused: {TEST_REFUSED} | aligned with D1 files: {ALIGN} | {time.time()-t0:.0f} s')
if EXPECTED['train_windows'] > 0: assert len(BTR.win) == EXPECTED['train_windows'] and len(BVA.win) == EXPECTED['val_windows'], 'window counts differ from the protocol'

train 3,453,034 windows | validation 1,725,989 windows (early-stopping subset 431,497) | test days refused: True | aligned with D1 files: True | 64 s


## 6 · Tune M1 and M0 (2 configurations each, one seed, validation subset). Stops itself after `MAX_RUN_MINUTES`.

In [9]:
DEADLINE = time.time() + MAX_RUN_MINUTES * 60; INCOMPLETE = False; DONE = {}
def run(make, cfg, tag, seed):
    global INCOMPLETE
    if INCOMPLETE: return None
    print(f'[{tag}] cfg {cfg} seed {seed}')
    d = L.fit2(make, cfg, BTR, BVA, VALSUB, OUT / 'runs' / tag, seed=seed, frac=FRAC, max_epochs=MAX_EPOCHS, patience=PATIENCE, bs=BATCH, device=DEV, amp=AMP, deadline=DEADLINE, aux=AUX)
    if d is None: INCOMPLETE = True
    else: DONE[tag] = d
    return d
MAKE = {'M1': L.make_m1, 'M0': L.make_m0, 'B3a': L.make_mlpa}
TUNING = {'M1': [], 'M0': []}; BEST_CFG = {'B3a': B3_CFG}
for name in ('M1', 'M0'):
    for i, cfg in enumerate(GRID):
        d = run(MAKE[name], cfg, f'tune_{name}_{i}', SEED_TUNE)
        if d is not None: TUNING[name].append({'config': i, **cfg, 'best_val_sub': d['best_val_sub'], 'best_epoch': d['best_epoch'], 'epochs_run': d['epochs_run'], 'mean_epoch_seconds': d['mean_epoch_seconds'], 'n_params': d['n_params']})
    if len(TUNING[name]) == len(GRID):
        df = pd.DataFrame(TUNING[name]); display(df.round(6)); b = int(df['best_val_sub'].idxmin()); BEST_CFG[name] = GRID[b]; print(name, 'best configuration (validation subset only):', b, GRID[b])
if not INCOMPLETE: S.put_json({'best': BEST_CFG, 'tuning': TUNING}, OUT / 'tuning.json')
else: print('\n>>> TIME BUDGET REACHED. Nothing is lost: press Run all again (Runtime menu) and training continues from the last finished epoch. <<<')

[tune_M1_0] cfg {'hidden': 64, 'lr': 0.001} seed 200
[tune_M1_1] cfg {'hidden': 64, 'lr': 0.0003} seed 200


,config,hidden,lr,best_val_sub,best_epoch,epochs_run,mean_epoch_seconds,n_params
0,0,64,0.0010,0.149405,11,14,356.485714,76955
1,1,64,0.0003,0.149507,12,12,354.000000,76955


M1 best configuration (validation subset only): 0 {'hidden': 64, 'lr': 0.001}
[tune_M0_0] cfg {'hidden': 64, 'lr': 0.001} seed 200
[tune_M0_1] cfg {'hidden': 64, 'lr': 0.0003} seed 200


,config,hidden,lr,best_val_sub,best_epoch,epochs_run,mean_epoch_seconds,n_params
0,0,64,0.0010,0.150337,6,9,257.2,60571
1,1,64,0.0003,0.150267,12,12,259.3,60571


M0 best configuration (validation subset only): 1 {'hidden': 64, 'lr': 0.0003}


## 7 · Final training: 3 seeds each for M1, M0 and B3a. Stops itself after `MAX_RUN_MINUTES`; run all again to continue.

In [10]:
assert not INCOMPLETE, 'Time budget reached in the tuning stage: press Run all again to continue (finished epochs are kept).'
FINAL = {'M1': {}, 'M0': {}, 'B3a': {}}
for name in ('M1', 'M0', 'B3a'):
    for sd in SEEDS:
        d = run(MAKE[name], BEST_CFG[name], f'final_{name}_{sd}', sd)
        if d is not None: FINAL[name][sd] = d
if INCOMPLETE: print('\n>>> TIME BUDGET REACHED. Nothing is lost: press Run all again (Runtime menu) and training continues from the last finished epoch. <<<')
else:
    rows = [{'model': n, 'seed': s, 'best epoch': d['best_epoch'], 'epochs run': d['epochs_run'], 'val MAE (25% subset)': d['best_val_sub'], 'epoch s': d['mean_epoch_seconds'], 'params': d['n_params']} for n in FINAL for s, d in FINAL[n].items()]
    display(pd.DataFrame(rows).round(6))

[final_M1_201] cfg {'hidden': 64, 'lr': 0.001} seed 201
[final_M1_202] cfg {'hidden': 64, 'lr': 0.001} seed 202
[final_M1_203] cfg {'hidden': 64, 'lr': 0.001} seed 203
[final_M0_201] cfg {'hidden': 64, 'lr': 0.0003} seed 201
[final_M0_202] cfg {'hidden': 64, 'lr': 0.0003} seed 202
[final_M0_203] cfg {'hidden': 64, 'lr': 0.0003} seed 203
  epoch 1: train L1 0.16105 | val(sub) 0.15203 | best 0.15203 (e1) | 245.4 s
  epoch 2: train L1 0.15554 | val(sub) 0.15066 | best 0.15066 (e2) | 244.8 s
  epoch 3: train L1 0.15440 | val(sub) 0.15040 | best 0.15040 (e3) | 244.7 s
  epoch 4: train L1 0.15429 | val(sub) 0.15029 | best 0.15029 (e4) | 244.4 s
  epoch 5: train L1 0.15369 | val(sub) 0.15023 | best 0.15023 (e5) | 245.1 s
  epoch 6: train L1 0.15397 | val(sub) 0.15013 | best 0.15013 (e6) | 243.9 s
  epoch 7: train L1 0.15354 | val(sub) 0.15020 | best 0.15013 (e6) | 244.4 s
  epoch 8: train L1 0.15407 | val(sub) 0.15018 | best 0.15013 (e6) | 243.8 s
  epoch 9: train L1 0.15407 | val(sub) 0.1501

,model,seed,best epoch,epochs run,val MAE (25% subset),epoch s,params
0,M1,201,12,12,0.149340,356.458333,76955
1,M1,202,11,12,0.149584,346.691667,76955
2,M1,203,9,12,0.149505,340.091667,76955
3,M0,201,11,12,0.150058,247.308333,60571
4,M0,202,9,12,0.149996,246.233333,60571
5,M0,203,6,9,0.150135,244.577778,60571
6,B3a,201,11,12,0.153194,9.950000,980389
7,B3a,202,2,5,0.153282,9.980000,980389
8,B3a,203,7,10,0.153176,9.900000,980389


## 8 · Score on all validation windows (seeds averaged per window, as in the frozen rule) and compare descriptively with B1 and B3

In [11]:
assert not INCOMPLETE, 'Training is not finished: press Run all again to continue (finished epochs are kept).'
t0 = time.time(); RES = {}; AUCS = {}
for name in ('M1', 'M0', 'B3a'):
    npz = OUT / f'{name}_val_seed_averaged.npz'; jf = OUT / f'score_{name}.json'
    if npz.exists() and jf.exists():
        z = np.load(npz); RES[name] = {k: z[k] for k in ('loss', 'chg_sum', 'n_chg', 'spr')}; AUCS[name] = json.loads(jf.read_text()); print(name, 'already scored - loaded'); continue
    rs = []; ca = []
    for sd in SEEDS:
        m, _ = L.load_best2(MAKE[name], BEST_CFG[name], OUT / 'runs' / f'final_{name}_{sd}', DEV)
        rs.append(S.evaluate(BVA, L.predict_fn(m, DEV, AMP), bs=256)); ca.append(L.change_auc(m, BVA, AUCW, DEV, AMP)); del m
    RES[name] = S.average_results(rs); AUCS[name] = {'per_seed': ca, 'auc': float(np.mean([c['auc'] for c in ca])), 'bce': float(np.mean([c['bce'] for c in ca])), 'base_rate': ca[0]['base_rate']}
    S.save_result(RES[name], BVA, npz); S.dump_json(AUCS[name], jf); print(f'{name} scored on all validation windows ({time.time()-t0:.0f} s)')
def pooled(r, mask=None):
    m = np.ones(len(r['loss']), bool) if mask is None else mask
    return {'all': float(r['loss'][m].mean()), 'changed': float(r['chg_sum'][m].sum() / max(int(r['n_chg'][m].sum()), 1)), 'spread': float(r['spr'][m].mean())}
ALLR = {'B1': R_B1, 'B3': R_B3, **RES}; SYMOF = np.array([k.split('_')[0] for k in BVA.keys])[BVA.win_set]
GROUPS = {'pooled': None, 'BTCUSDT': SYMOF == 'BTCUSDT', 'ETHUSDT': SYMOF == 'ETHUSDT'}
PO = {g: {n: pooled(r, m) for n, r in ALLR.items()} for g, m in GROUPS.items()}
gain = lambda a, b: 100 * (1 - a / b)
rows = []
for g in GROUPS:
    P = PO[g]; rows.append({'group': g, 'B1 all': P['B1']['all'], 'B3': P['B3']['all'], 'B3a': P['B3a']['all'], 'M0': P['M0']['all'], 'M1': P['M1']['all'], 'M1 vs B1 %': gain(P['M1']['all'], P['B1']['all']), 'M1 vs B3 %': gain(P['M1']['all'], P['B3']['all']), 'M1 vs M0 %': gain(P['M1']['all'], P['M0']['all']), 'M0 vs B1 %': gain(P['M0']['all'], P['B1']['all'])})
display(pd.DataFrame(rows).round(5))
rows = []
for g in GROUPS:
    P = PO[g]; rows.append({'group': g, 'B1 changed': P['B1']['changed'], 'M0 changed': P['M0']['changed'], 'M1 changed': P['M1']['changed'], 'M1 vs B1 % (changed)': gain(P['M1']['changed'], P['B1']['changed']), 'M1 vs M0 % (changed)': gain(P['M1']['changed'], P['M0']['changed']),
                 'B1 spread': P['B1']['spread'], 'M1 spread': P['M1']['spread'], 'M1 vs B1 % (spread)': gain(P['M1']['spread'], P['B1']['spread'])})
display(pd.DataFrame(rows).round(5))
display(pd.DataFrame({n: {'AUC (will a node change?)': AUCS[n]['auc'], 'BCE': AUCS[n]['bce'], 'base rate': AUCS[n]['base_rate']} for n in AUCS}).T.round(4))
print(f'(D1b shared-node reference, validation: ' + (f'{D1B_REP["shared"]["pooled"]["rel_all_pct"]:+.2f}% all-target gain, AUC {D1B_REP["shared"]["pooled"]["auc_change"]:.3f})' if D1B_REP else 'D1b report not found)'))

M1 scored on all validation windows (1389 s)
M0 scored on all validation windows (2394 s)
B3a scored on all validation windows (2421 s)


,group,B1 all,B3,B3a,M0,M1,M1 vs B1 %,M1 vs B3 %,M1 vs M0 %,M0 vs B1 %
0,pooled,0.15275,0.15277,0.15316,0.14998,0.14939,2.19702,2.21390,0.39120,1.81291
1,BTCUSDT,0.04560,0.04563,0.04593,0.04374,0.04361,4.38472,4.44051,0.30717,4.09011
2,ETHUSDT,0.25996,0.25999,0.26046,0.25629,0.25525,1.81294,1.82283,0.40551,1.41316


,group,B1 changed,M0 changed,M1 changed,M1 vs B1 % (changed),M1 vs M0 % (changed),B1 spread,M1 spread,M1 vs B1 % (spread)
0,pooled,0.70346,0.68089,0.67251,4.39992,1.23141,0.02674,0.01817,32.05682
1,BTCUSDT,0.21728,0.20232,0.20082,7.57415,0.74219,0.04040,0.02691,33.38162
2,ETHUSDT,1.15852,1.12883,1.11400,3.84271,1.31353,0.01306,0.00941,27.95684


,AUC (will a node change?),BCE,base rate
M1,0.7543,0.4423,0.2124
M0,0.7511,0.4438,0.2124
B3a,0.7049,0.4677,0.2124


(D1b shared-node reference, validation: +0.84% all-target gain, AUC 0.732)


## 9 · Reading rule (written before the numbers)

In [12]:
# ---- reading rule, written before the numbers were seen (descriptive, validation; the real H1/H2 test is D3) ----
P = PO['pooled']; g_b1 = gain(P['M1']['all'], P['B1']['all']); g_b3 = gain(P['M1']['all'], P['B3']['all']); g_m0 = gain(P['M1']['all'], P['M0']['all']); a1 = AUCS['M1']['auc']; a0 = AUCS['M0']['auc']; ab = AUCS['B3a']['auc']
if g_b1 >= 5.0 and g_b3 >= 5.0 and g_m0 >= 5.0: READ = 'VALIDATION PASS-LEVEL: M1 clears 5% vs persistence, vs B3 and vs M0 on validation. This is promising but is NOT a test result; D3 decides.'
elif g_b1 >= 5.0 and g_b3 >= 5.0: READ = f'M1 clears 5% vs persistence and vs B3 but only {g_m0:+.2f}% vs M0: the gain comes from the temporal/shared design, not from graph edges.'
elif g_m0 >= 1.0: READ = f'M1 is {g_m0:+.2f}% better than M0 (graph edges help a little) but does not clear 5% vs persistence ({g_b1:+.2f}%): H1 likely fails, H2 likely fails or is marginal.'
else: READ = f'M1 is {g_b1:+.2f}% vs persistence, {g_b3:+.2f}% vs B3 and {g_m0:+.2f}% vs M0: no evidence for H1 or H2 on validation. If the test days agree, both will be reported as failed.'
AUC_READ = f'change occurrence AUC: M1 {a1:.3f} | M0 {a0:.3f} | B3a {ab:.3f}' + (f' | D1b shared-node net {D1B_REP["shared"]["pooled"]["auc_change"]:.3f}' if D1B_REP else '')
print(READ); print(AUC_READ)

M1 is +2.20% vs persistence, +2.21% vs B3 and +0.39% vs M0: no evidence for H1 or H2 on validation. If the test days agree, both will be reported as failed.
change occurrence AUC: M1 0.754 | M0 0.751 | B3a 0.705 | D1b shared-node net 0.732


## 10 · Saved report (never overwritten) and the SHARE BLOCK

In [13]:
assert not INCOMPLETE, 'Not finished: press Run all again.'
RT = []
RT.append(('R01', 'all 3 models x 3 seeds finished with finite validation MAE', all(len(FINAL[n]) == len(SEEDS) for n in FINAL) and all(np.isfinite(d['best_val_sub']) for n in FINAL for d in FINAL[n].values()), ''))
RT.append(('R02', 'validation windows equal the D1 result files (same order)', ALIGN, ''))
RT.append(('R03', 'no test day was opened (loader refused, no freeze file)', TEST_REFUSED and not FREEZE_FILE.exists(), ''))
RT.append(('R04', 'every learned model used the same budget (fraction, epochs, patience, batch, loss weight, validation subset)', all(d['frac'] == FRAC and d['max_epochs'] == MAX_EPOCHS and d['patience'] == PATIENCE and d['batch_size'] == BATCH and d['aux_weight'] == AUX for n in FINAL for d in FINAL[n].values()), ''))
RT.append(('R05', 'M1 and M0 differ only in the neighbour edges (same hidden size rule, same grid)', True, f'M1 {BEST_CFG["M1"]} | M0 {BEST_CFG["M0"]} | B3a {BEST_CFG["B3a"]}'))
ALL = [(r[0], r[1], bool(r[2]), r[3]) for r in ER] + [(r[0], r[1], bool(r[2]), r[3]) for r in RT]; ok = all(r[2] for r in ALL)
VERDICT = 'D2 COMPLETE' if ok else 'D2 INCOMPLETE (a check failed)'
ep = {n: float(np.mean([d['mean_epoch_seconds'] for d in FINAL[n].values()])) for n in FINAL}; ne = {n: float(np.mean([d['epochs_run'] for d in FINAL[n].values()])) for n in FINAL}
rep = {'verdict': VERDICT, 'read': READ, 'auc_read': AUC_READ, 'tests': ALL, 'best_config': BEST_CFG, 'tuning': TUNING, 'final': {n: {str(s): {k: v for k, v in d.items() if k != 'history'} for s, d in FINAL[n].items()} for n in FINAL}, 'histories': {n: {str(s): d['history'] for s, d in FINAL[n].items()} for n in FINAL},
       'pooled_scores': PO, 'change_occurrence': AUCS, 'd1b_reference': (D1B_REP['shared']['pooled'] if D1B_REP else None), 'epoch_seconds': ep, 'epochs_run': ne, 'frozen': FROZEN, 'device': HW, 'test_days_opened': False, 'note': 'validation only; descriptive; H1/H2 are decided once in D3'}
pth = S.put_json(rep, OUT / 'd2_report.json'); print('report:', pth.relative_to(BASE))
f = lambda v: f'{v:+.2f}%'
LN = ['==== D2 SHARE BLOCK (copy everything between the two ==== lines) ====', f'{VERDICT} | tests passed {sum(r[2] for r in ALL)}/{len(ALL)} | gpu {GPU} | test days opened: NO | lib {LIB_SHA[:12]} / {D2LIB_SHA[:12]}', READ, AUC_READ,
      f'configs: M1 {BEST_CFG["M1"]} | M0 {BEST_CFG["M0"]} | B3a {BEST_CFG["B3a"]}', f'mean epoch seconds: ' + ', '.join(f'{n} {ep[n]:.0f}' for n in ep) + ' | mean epochs run: ' + ', '.join(f'{n} {ne[n]:.1f}' for n in ne),
      'tuning (val subset MAE): ' + '; '.join(f'{n} ' + ', '.join(f'c{r["config"]} {r["best_val_sub"]:.6f}' for r in TUNING[n]) for n in TUNING)]
for g in GROUPS:
    P = PO[g]; LN.append(f'  {g}: MAE all B1 {P["B1"]["all"]:.5f} | B3 {P["B3"]["all"]:.5f} | B3a {P["B3a"]["all"]:.5f} | M0 {P["M0"]["all"]:.5f} | M1 {P["M1"]["all"]:.5f} || M1 vs B1 {f(gain(P["M1"]["all"], P["B1"]["all"]))}, vs B3 {f(gain(P["M1"]["all"], P["B3"]["all"]))}, vs M0 {f(gain(P["M1"]["all"], P["M0"]["all"]))}')
    LN.append(f'      changed-node MAE B1 {P["B1"]["changed"]:.5f} | M0 {P["M0"]["changed"]:.5f} | M1 {P["M1"]["changed"]:.5f} (M1 vs B1 {f(gain(P["M1"]["changed"], P["B1"]["changed"]))}, vs M0 {f(gain(P["M1"]["changed"], P["M0"]["changed"]))}) | spread MAE B1 {P["B1"]["spread"]:.5f} | M1 {P["M1"]["spread"]:.5f} ({f(gain(P["M1"]["spread"], P["B1"]["spread"]))})')
LN.append('failed tests: ' + (', '.join(r[0] for r in ALL if not r[2]) or 'none')); LN.append('==== END ====')
print('\n'.join(LN)); (OUT / 'd2_share_block.txt').write_text('\n'.join(LN)) if not (OUT / 'd2_share_block.txt').exists() else None

report: TGNN_StageD_v1/d2/d2_report.json
==== D2 SHARE BLOCK (copy everything between the two ==== lines) ====
D2 COMPLETE | tests passed 27/27 | gpu Tesla T4 | test days opened: NO | lib 85e7c1f444a4 / 54b345bce8c3
M1 is +2.20% vs persistence, +2.21% vs B3 and +0.39% vs M0: no evidence for H1 or H2 on validation. If the test days agree, both will be reported as failed.
change occurrence AUC: M1 0.754 | M0 0.751 | B3a 0.705 | D1b shared-node net 0.732
configs: M1 {'hidden': 64, 'lr': 0.001} | M0 {'hidden': 64, 'lr': 0.0003} | B3a {'hidden': 64, 'lr': 0.0003}
mean epoch seconds: M1 348, M0 246, B3a 10 | mean epochs run: M1 12.0, M0 11.0, B3a 9.0
tuning (val subset MAE): M1 c0 0.149405, c1 0.149507; M0 c0 0.150337, c1 0.150267
  pooled: MAE all B1 0.15275 | B3 0.15277 | B3a 0.15316 | M0 0.14998 | M1 0.14939 || M1 vs B1 +2.20%, vs B3 +2.21%, vs M0 +0.39%
      changed-node MAE B1 0.70346 | M0 0.68089 | M1 0.67251 (M1 vs B1 +4.40%, vs M0 +1.23%) | spread MAE B1 0.02674 | M1 0.01817 (+32.06

1532